# 08 — Best baseline vs advanced model

## Goal

Compare matched experiment records for the validation-selected baseline and the advanced model across supported Davis/KIBA splits. Keep selection evidence distinct from final test reporting.


## Setup


In [ ]:
from pathlib import Path
from importlib import import_module
import json
import subprocess
import sys


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project checkout.")


def require_api(module_name: str, attribute: str):
    module = import_module(module_name)
    if not hasattr(module, attribute):
        raise RuntimeError(
            f"Required v2 API {module_name}.{attribute} is unavailable. "
            "Update v2-refactor and rerun the tests before enabling this cell."
        )
    return getattr(module, attribute)


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BRANCH = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if BRANCH != "v2-refactor":
    raise RuntimeError(f"Expected v2-refactor, found {BRANCH!r}. v1-final is read-only.")

print(f"Project: {PROJECT_ROOT}")
print(f"Branch:  {BRANCH}")


In [ ]:
from config import EXPERIMENT_LOG_PATH

SELECTED_BASELINE = None  # set to gcn/gat/gin only after notebook 06
ALLOW_PROTOCOL_DIFFERENCES = False  # change only with written justification
PROTOCOL_DIFFERENCE_JUSTIFICATION = ""
RUN_COMPARISON = False
print("Experiment log:", EXPERIMENT_LOG_PATH)


## Steps


In [ ]:
if RUN_COMPARISON:
    import pandas as pd

    if SELECTED_BASELINE not in {"gcn", "gat", "gin"}:
        raise ValueError(
            "Set SELECTED_BASELINE from notebook 06 before comparison."
        )
    records = pd.read_csv(EXPERIMENT_LOG_PATH)
    required = {
        "dataset", "split", "model", "seed", "split_fingerprint",
        "affinity_label", "configuration_used",
        "batch_size", "epochs", "learning_rate", "weight_decay",
        "best_validation_mse", "test_mse", "test_rmse", "test_mae",
        "concordance_index", "pearson_correlation",
    }
    missing = required.difference(records.columns)
    if missing:
        raise ValueError(f"Experiment log is missing columns: {sorted(missing)}")
    for column in ("dataset", "split", "model"):
        records[column] = records[column].astype(str).str.strip().str.lower()
    comparison = records[
        records["model"].isin([SELECTED_BASELINE, "advanced"])
    ].copy()
    if comparison.empty:
        raise ValueError("No baseline/advanced records were found.")
    if comparison["split_fingerprint"].isna().any() or (
        comparison["split_fingerprint"].astype(str).str.strip().eq("").any()
    ):
        raise ValueError("Every comparison record needs a split_fingerprint.")
    if comparison["affinity_label"].isna().any() or (
        comparison["affinity_label"].astype(str).str.strip().eq("").any()
    ):
        raise ValueError("Every comparison record needs an affinity_label.")

    keys = ["dataset", "split", "seed"]
    comparison["_source_order"] = range(len(comparison))
    if "timestamp" in comparison.columns:
        comparison["_timestamp_order"] = pd.to_datetime(
            comparison["timestamp"], errors="coerce", utc=True
        )
        comparison = comparison.sort_values(
            ["_timestamp_order", "_source_order"],
            kind="stable",
            na_position="first",
        )
    comparison = comparison.drop_duplicates(
        keys + ["model"], keep="last"
    )

    def parse_configuration(value):
        if isinstance(value, dict):
            return value
        if isinstance(value, str) and value.strip():
            try:
                parsed = json.loads(value)
            except json.JSONDecodeError as error:
                raise ValueError(
                    "configuration_used contains invalid JSON."
                ) from error
            if isinstance(parsed, dict):
                return parsed
        raise ValueError(
            "Every comparison record needs a JSON-object configuration_used."
        )

    configurations = comparison["configuration_used"].map(
        parse_configuration
    )
    configuration_fields = (
        "max_protein_length",
        "early_stopping_patience",
        "minimum_improvement",
        "optimizer",
        "loss_function",
        "checkpoint_selection_metric",
        "test_evaluation_policy",
    )
    for field in configuration_fields:
        values = configurations.map(lambda configuration: configuration.get(field))
        missing_values = values.map(
            lambda value: value is None
            or (isinstance(value, str) and not value.strip())
        )
        if missing_values.any():
            raise ValueError(
                f"configuration_used must record {field!r} for every run."
            )
        comparison[f"configuration__{field}"] = values

    baseline = comparison[comparison["model"].eq(SELECTED_BASELINE)]
    advanced = comparison[comparison["model"].eq("advanced")]
    baseline_keys = set(baseline[keys].itertuples(index=False, name=None))
    advanced_keys = set(advanced[keys].itertuples(index=False, name=None))
    if baseline_keys != advanced_keys:
        raise ValueError({
            "baseline_only_cells": sorted(baseline_keys - advanced_keys),
            "advanced_only_cells": sorted(advanced_keys - baseline_keys),
        })
    if not baseline_keys:
        raise ValueError("No matched baseline/advanced evidence cells exist.")

    matched = baseline.merge(
        advanced,
        on=keys,
        how="inner",
        validate="one_to_one",
        suffixes=("_baseline", "_advanced"),
    )
    fingerprint_mismatch = matched[
        matched["split_fingerprint_baseline"]
        != matched["split_fingerprint_advanced"]
    ]
    if not fingerprint_mismatch.empty:
        raise ValueError(
            "Baseline/advanced split fingerprints differ for cells: "
            f"{fingerprint_mismatch[keys].to_dict('records')}"
        )

    affinity_mismatch = matched[
        matched["affinity_label_baseline"]
        != matched["affinity_label_advanced"]
    ]
    if not affinity_mismatch.empty:
        raise ValueError(
            "Baseline/advanced affinity labels differ for cells: "
            f"{affinity_mismatch[keys].to_dict('records')}"
        )

    protocol_columns = [
        "batch_size", "epochs", "learning_rate", "weight_decay",
        *[
            f"configuration__{field}"
            for field in configuration_fields
        ],
    ]
    differences = []
    for column in protocol_columns:
        mismatch = matched[
            matched[f"{column}_baseline"] != matched[f"{column}_advanced"]
        ]
        for row in mismatch.itertuples(index=False):
            differences.append({
                **{key: getattr(row, key) for key in keys},
                "field": column,
                "baseline": getattr(row, f"{column}_baseline"),
                "advanced": getattr(row, f"{column}_advanced"),
            })
    protocol_differences = pd.DataFrame(differences)
    if not protocol_differences.empty:
        print("Protocol differences:\n", protocol_differences.to_string(index=False))
        if not ALLOW_PROTOCOL_DIFFERENCES:
            raise ValueError(
                "Protocol differences require written justification and "
                "ALLOW_PROTOCOL_DIFFERENCES=True."
            )
        if not PROTOCOL_DIFFERENCE_JUSTIFICATION.strip():
            raise ValueError(
                "Set PROTOCOL_DIFFERENCE_JUSTIFICATION before allowing "
                "training/optimisation protocol differences."
            )
        print(
            "Protocol-difference justification:",
            PROTOCOL_DIFFERENCE_JUSTIFICATION,
        )

    columns = keys + [
        "best_validation_mse_baseline", "best_validation_mse_advanced",
        "test_mse_baseline", "test_mse_advanced",
        "test_rmse_baseline", "test_rmse_advanced",
        "test_mae_baseline", "test_mae_advanced",
        "concordance_index_baseline", "concordance_index_advanced",
        "pearson_correlation_baseline", "pearson_correlation_advanced",
    ]
    print(matched[columns].sort_values(keys).to_string(index=False))
else:
    print("Comparison skipped: RUN_COMPARISON is False.")


## Checks

Duplicate reruns are reduced to the latest timestamp (or last row when unavailable). Baseline and advanced records must have identical dataset/split/seed cells, affinity labels, and exact split fingerprints; these identity/provenance mismatches can never be overridden. Batch size, epoch budget, learning rate, weight decay, maximum protein length, early-stopping settings, optimiser, loss, checkpoint-selection metric, and test-evaluation policy are surfaced and fail closed unless explicitly allowed with a written protocol justification. Davis and KIBA rows remain separate—raw MSE values are never aggregated across their incompatible scales. The advanced model must not be called universally “pKd”; use the dataset-specific interpretation.

## Next steps

Select checkpoint references using validation evidence before running AML screening.
